# Classifier Free Guidance
我们已经掌握了 Classifier Guidance(CG) 的原理和核心代码的实现，虽然 CG 实现了对扩散模型生成图像类别的控制，但是它也有一定的缺点，比如需要额外训练一个分类器，并且分类器训练完成之后，就只能控制模型生成固定类别的图像，如果要添加新的类别，就需要重新训练分类器了，而本篇的 **Classifier-Free Guidance(CFG)** 就能解决上述的两个问题

<div align="center">
    <img src="./imgs/cfg_example.png" alt="cfg_example" style="width: 450px; height: auto;">
</div>

本篇notebook基于如下链接的内容和代码进行整理，参考链接：

[参考知乎解析](https://zhuanlan.zhihu.com/p/712821377)

[CFG 论文](https://arxiv.org/pdf/2207.12598)

[CFG Pytorch 实现](https://github.com/jcwang-gh/classifier-free-diffusion-guidance-Pytorch/blob/master/diffusion.py)

## CFG理论回顾
在[理论推导](./theory.md)部分，我们已经知道了，最后的 CFG 公式如下:

$$
\epsilon_\theta(x_t|c)^{\text{CFG}} = \epsilon_\theta(x_t|\emptyset) + w \cdot \left( \epsilon_\theta(x_t|c) - \epsilon_\theta(x_t|\emptyset) \right)
$$

直观理解，就是在原有的无条件生成的基础上，往有条件的方向$\left( \epsilon_\theta(x_t|c) - \epsilon_\theta(x_t|\emptyset) \right)$进行修正，而$w$则控制了修正的力度，下面我们就根据这个公式，对 DDPM 模型进行一些修改，从而实现 CFG

## 类别标签嵌入
为了实现类别标签嵌入，让模型能够根据类别标签进行预测，我们就需要实现一个类别嵌入的类，假设有两个类别，那类别标签总共就有三个: 0、1表示我们的两个类别，2表示无类别条件，用来进行无条件预测；

并且为了实现无条件预测，在训练时需要对类别标签进行随机的丢弃，比如(img1, 0)就变成(img1, 2)，从而让模型学会进行无条件预测

In [3]:
import torch
import torch.nn as nn


class ClassEmbedding(nn.Module):
    """
    Embedding for class label c
    """
    def __init__(self, n_channels: int, n_classes, token_dropout_prob: float):
        super().__init__()
        
        # 增加空类别(最后一类)
        self.embedding_table = nn.Embedding(n_classes+1, n_channels)  
        self.n_classes = n_classes
        self.token_dropout_prob = token_dropout_prob

    def token_drop(self, labels, force_drop_ids=None):
        """
        实现类别标签丢弃，以支持CFG
        
        * labels: e.g. [0, 1, 2]
        * force_drop_ids: e.g. [1, 0, 0]，1的位置要丢弃
        """
        # 确定丢弃位置，如[True, False]，丢弃True位置的标签，设为空类别
        if force_drop_ids is None:
            drop_ids = torch.rand(labels.shape[0], device=labels.device) < self.token_dropout_prob
        else:
            drop_ids = force_drop_ids == 1

        print(f'Before drop, labels: {labels}')

        # True就选空类，否则选对应位置的label
        labels = torch.where(drop_ids, self.n_classes, labels)

        print(f'After drop, labels: {labels}')
        
        return labels 
    
    def forward(self, labels, if_train: bool, force_drop_ids=None):
        if (if_train) or (force_drop_ids is not None):
            # 在训练或者是要实现无条件预测时，就要进行类别标签的丢弃
            labels = self.token_drop(labels, force_drop_ids)

        emb = self.embedding_table(labels)

        return emb
    

In [ ]:
# 类别标签变成10，就说明丢弃为无条件了
n_classes = 10
c = torch.randint(0, n_classes, (10, ))


CE = ClassEmbedding(512, n_classes, 0.2)
class_emb = CE(c, True)
print(f'After embedding, class_emb size: {class_emb.size()}')


Before drop, labels: tensor([2, 6, 5, 7, 3, 4, 6, 7, 7, 4])
After drop, labels: tensor([ 2,  6,  5,  7, 10,  4,  6,  7,  7,  4])
After embedding, class_emb size: torch.Size([10, 512])


## 进行条件嵌入
我们得到类别标签嵌入之后，就要把条件注入到模型中，一种最简单的修改方式就是直接和时间嵌入$t$相加，这样就能实现最小改动，也是我们本篇代码中使用的方式:

```python
...
t_emb = self.t_emb(t)
c_emb = self.c_emb(c)
cond_emb = t_emb + c_emb
...
```

当然，在实际使用中，还有其他方式，比如交叉注意力、通道注意力、自适应层归一化等等



### 交叉注意力
就是 $Q$ 来自图像特征，$K/V$来自条件特征，输入进行注意力计算，从而把条件特征融入到图像特征之中，下面我们简单实现一个交叉注意力

In [20]:
class CrossAttention(nn.Module):
    """简易交叉注意力，实际使用以官方实现为准"""
    def __init__(self, dim=768):
        super().__init__()

        self.dim = dim

        self.q = nn.Linear(3, dim)
        self.k = nn.Linear(dim, dim)
        self.v = nn.Linear(dim, dim)

        self.out = nn.Linear(dim, 3)

    def forward(self, x: torch.Tensor, cond: torch.Tensor):
        batch_size, n_channels, h, w = x.shape

        # [bs, dim, h, w] -> [bs, dim, h*w] -> [bs, h*w, dim]
        x = x.view(batch_size, n_channels, -1).permute(0, 2, 1)

        # [bs, dim] -> [bs, 1, dim]
        cond = cond.unsqueeze(1)

        Q = self.q(x)
        K = self.k(cond)
        V = self.v(cond)

        # softmax(QK^T / sqrt(d))
        attn = torch.bmm(Q, K.transpose(-2,-1)) / (self.dim**0.5)
        attn = attn.softmax(dim=-1)

        # *V
        out = torch.bmm(attn, V)
        out = self.out(out) + x

        # [bs, h*w, dim] -> [bs, dim, h*w] -> [bs, dim, h, w]
        out = out.permute(0, 2, 1).view(batch_size, n_channels, h, w)

        return out


In [21]:
batch_size = 4
x = torch.randn(batch_size, 3, 64, 64)
c = torch.randint(0, 10, (batch_size, ))

print(f'Before Cross Atten, x size: {x.size()}')
CE = ClassEmbedding(768, 10, 0.2)
c_emb = CE(c, True)
CA = CrossAttention()
x = CA(x, c_emb)
print(f'After Cross Atten, x size: {x.size()}')


Before Cross Atten, x size: torch.Size([4, 3, 64, 64])
Before drop, labels: tensor([5, 1, 8, 2])
After drop, labels: tensor([ 5, 10,  8,  2])
After Cross Atten, x size: torch.Size([4, 3, 64, 64])


### 通道注意力
通道注意力实际上是在 cond 进行一层线性投影层之后，直接和 t_emb 相加，比交叉注意力容易，一个简易实现如下:

In [25]:
class ChannelAttention(nn.Module):
    """简易通道注意力，实际使用以官方实现为准"""
    def __init__(self, dim):
        super().__init__()

        self.proj = nn.Linear(dim, dim)

    def forward(self, t_emb, class_emb):
        class_cond = self.proj(class_emb)
        
        cond = t_emb + class_cond
        
        return cond


In [ ]:
t_emb = torch.randn(4, 512)
c_emb = torch.randn(4, 512)


print(f'Before Channel Atten, t_emb size: {t_emb.size()}')
print(f'Before Channel Atten, c_emb size: {c_emb.size()}')
CA = ChannelAttention(512)
cond_emb = CA(t_emb, c_emb)
print(f'After Channel Atten, cond_emb size: {cond_emb.size()}')


Before Channel Atten, t_emb size: torch.Size([4, 512])
Before Channel Atten, c_emb size: torch.Size([4, 512])
After Channel Atten, cond_emb size: torch.Size([4, 512])


### 自适应归一化
这里以 AdaLN (自适应层归一化)为例进行实现，实际上就是对输入进行归一化之后，使用条件嵌入经过 MLP 预测得到 scale 和 shift 系数进行仿射变换，简易实现如下:

In [42]:
class AdaLayerNorm(nn.Module):
    """简易AdaLN，实际使用以官方实现为准"""
    def __init__(self, input_dim=64, mlp_ratio=4, n_classes=10):
        super().__init__()

        self.emb = nn.Embedding(n_classes+1, input_dim)
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, input_dim*mlp_ratio),
            nn.SiLU(),
            nn.Linear(input_dim*mlp_ratio, input_dim*2),
        )
        # 只做纯归一化，不学习放缩和偏移，而是由条件嵌入预测得到
        self.norm = nn.LayerNorm(input_dim, elementwise_affine=False)

    def forward(self, x, class_labels):
        c_emb = self.emb(class_labels)  # [bs,] -> [bs, input_dim]
        # [bs, input_dim] -> 2*[bs, 1, input_dim]
        scale, shift = torch.chunk(self.mlp(c_emb), 2, dim=-1)
        scale = scale.unsqueeze(1)
        shift = shift.unsqueeze(1)

        # print(scale.size())
        # print(shift.size())
        # print(x.size())
        # 运算时发生了广播，以scale为例
        # [bs, 1, input_dim] -> [bs, seq_len, input_dim]，然后逐元素乘法
        x = self.norm(x)*(1+scale) + shift

        return x


In [45]:
ALN = AdaLayerNorm()
c = torch.randint(0, 11, (4, ))
x = torch.randn(4, 114, 64)

print(f'Before AdaLN, x size: {x.size()}')
print(f'Before AdaLN, class_labels: {c}')
x = ALN(x, c)
print(f'After AdaLN, x size: {x.size()}')

Before AdaLN, x size: torch.Size([4, 114, 64])
Before AdaLN, class_labels: tensor([6, 1, 9, 1])
After AdaLN, x size: torch.Size([4, 114, 64])


## CFG 采样的实现
就是每一步采样要进行两次预测，有条件预测和无条件预测，再根据下面的公式进行条件引导控制类别:

$$
\epsilon_\theta(x_t|c)^{\text{CFG}} = \epsilon_\theta(x_t|\emptyset) + w \cdot \left( \epsilon_\theta(x_t|c) - \epsilon_\theta(x_t|\emptyset) \right)
$$

下面就是核心代码:

In [ ]:
from tqdm import tqdm
import os 
from torchvision.utils import save_image


def gather(consts: torch.Tensor, t: torch.Tensor):
    """Gather consts for $t$ and reshape to feature map shape"""
    c = consts.gather(-1, t)  # 取出t对应的噪声调度、α或α的连乘
    
    # -> [bs, 1, 1, 1]，运算时广播为[bs, C, H, W]
    return c.reshape(-1, 1, 1, 1) 

def p_sample_with_cfg(self, xt: torch.Tensor, t: torch.Tensor, c: torch.Tensor):
        """CFG单步采样"""
        # -------------------------------------------------------------------------------------
        # CFG引导实现
        # -------------------------------------------------------------------------------------
        eps_cond = self.eps_model(xt, t, c, False)
        force_drop_idx = torch.ones(c.shape[0], dtype=torch.long, device=self.device)
        eps_uncond = self.eps_model(xt, t, c, False, force_drop_idx)

        eps_theta = eps_uncond + self.guidance_scale * (eps_cond - eps_uncond)
        

        # -------------------------------------------------------------------------------------
        # 使用引导之后的预测噪声进行采样
        # -------------------------------------------------------------------------------------
        # [gather](utils.html) $\bar\alpha_t$
        alpha_bar = gather(self.alpha_bar, t)
        # $\alpha_t$
        alpha = gather(self.alpha, t)
        # $\frac{\beta}{\sqrt{1-\bar\alpha_t}}$
        eps_coef = (1 - alpha) / (1 - alpha_bar) ** .5
        # $$\frac{1}{\sqrt{\alpha_t}} \Big(x_t -
        #      \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\textcolor{lightgreen}{\epsilon_\theta}(x_t, t) \Big)$$
        mean = 1 / (alpha ** 0.5) * (xt - eps_coef * eps_theta)
        # $\sigma^2$
        var = gather(self.sigma2, t)

        # $\epsilon \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$
        eps = torch.randn(xt.shape, device=xt.device)
        
        # Sample，根据公式结合模型预测的噪声，求出的均值和方差，之后采样得到x_{t-1}
        return mean + (var ** .5) * eps

def sample_with_cfg(self, save_dir, c, sample_num=1, img_size=64, img_channels=3):
        """CFG采样"""
        self.eps_model.eval()

        with torch.no_grad():
            # 每一步采样，需要两次预测: 带条件和不带条件
            x = torch.randn([sample_num, img_channels, img_size, img_size], device=self.device)

            for t_ in tqdm(range(self.n_steps), desc=f"Sampling with CFG, class: {c}"): 
                t = self.n_steps - t_ - 1
                x = self.p_sample_with_cfg(x, x.new_full((sample_num,), t, dtype=torch.long), 
                                           x.new_full((sample_num,), c, dtype=torch.long))
                
            # DDPM输入和输出在-1~1之间，在此之前需要转为0~1才能存为图像
            x = (x + 1) / 2.0
            x = x.clamp(0.0, 1.0)

            os.makedirs(save_dir, exist_ok=True)
            for i in range(sample_num):  
                save_image(
                    x[i],
                    f"{save_dir}/sample_{i}.png",
                    normalize=False
                )

                print(f"Saved sample_{i}.png to {save_dir}")


## 示例

In [48]:
from c_cfg import (
    get_config,
    get_ddpm_cfg,
    CFGDDPM_trainer
)


In [ ]:
config = get_config('./configs/ddpm_cfg_unt_config.yaml')
config["train"]["epochs"] = 500  # 500 epochs示例

ddpm_cfg = get_ddpm_cfg(config, False)
trainer = CFGDDPM_trainer(ddpm_cfg, config)
trainer.train()


Training: 100%|██████████| 500/500 [01:59<00:00,  4.20it/s, avg_loss=0.0739]


In [50]:
ddpm_cfg_pretrained = get_ddpm_cfg(config, True)
ddpm_cfg_pretrained.sample_with_cfg(save_dir="cfg_0", c=0, sample_num=4)
ddpm_cfg_pretrained.sample_with_cfg(save_dir="cfg_1", c=1, sample_num=4)
ddpm_cfg_pretrained.sample("no_cfg", sample_num=4)


Loaded checkpoint from ckpts\cfg_ddpm_unet_iter50000.pth


Sampling with CFG, class: 0: 100%|██████████| 1000/1000 [00:56<00:00, 17.81it/s]


Saved sample_0.png to cfg_0
Saved sample_1.png to cfg_0
Saved sample_2.png to cfg_0
Saved sample_3.png to cfg_0


Sampling with CFG, class: 1: 100%|██████████| 1000/1000 [00:53<00:00, 18.63it/s]


Saved sample_0.png to cfg_1
Saved sample_1.png to cfg_1
Saved sample_2.png to cfg_1
Saved sample_3.png to cfg_1


Sampling without CFG: 100%|██████████| 1000/1000 [00:26<00:00, 37.07it/s]

Saved sample_0.png to no_cfg
Saved sample_1.png to no_cfg
Saved sample_2.png to no_cfg
Saved sample_3.png to no_cfg


可以展示一下结果:

<div style="margin: 24px 0;">
  <div style="text-align: center; font-size: 16px; font-weight: bold; margin-bottom: 12px;">class 0: 高松灯企鹅</div>
  <div style="display: flex; justify-content: center; gap: 12px; align-items: center; margin: 16px 0;">
    <div style="flex: 1; max-width: 120px; text-align: center;">
      <img src="./cfg_0/sample_0.png" style="width: 100%; height: auto;" alt="img1">
      <p style="margin: 4px 0 0; font-size: 14px;">图 1</p>
    </div>
    <div style="flex: 1; max-width: 120px; text-align: center;">
      <img src="./cfg_0/sample_1.png" style="width: 100%; height: auto;" alt="img2">
      <p style="margin: 4px 0 0; font-size: 14px;">图 2</p>
    </div>
    <div style="flex: 1; max-width: 120px; text-align: center;">
      <img src="./cfg_0/sample_2.png" style="width: 100%; height: auto;" alt="img3">
      <p style="margin: 4px 0 0; font-size: 14px;">图 3</p>
    </div>
    <div style="flex: 1; max-width: 120px; text-align: center;">
      <img src="./cfg_0/sample_3.png" style="width: 100%; height: auto;" alt="img4">
      <p style="margin: 4px 0 0; font-size: 14px;">图 4</p>
    </div>
  </div>
</div>

<div style="margin: 24px 0;">
  <div style="text-align: center; font-size: 16px; font-weight: bold; margin-bottom: 12px;">class 1: 菲比企鹅</div>
  <div style="display: flex; justify-content: center; gap: 12px; align-items: center; margin: 16px 0;">
    <div style="flex: 1; max-width: 120px; text-align: center;">
      <img src="./cfg_1/sample_0.png" style="width: 100%; height: auto;" alt="img1">
      <p style="margin: 4px 0 0; font-size: 14px;">图 1</p>
    </div>
    <div style="flex: 1; max-width: 120px; text-align: center;">
      <img src="./cfg_1/sample_1.png" style="width: 100%; height: auto;" alt="img2">
      <p style="margin: 4px 0 0; font-size: 14px;">图 2</p>
    </div>
    <div style="flex: 1; max-width: 120px; text-align: center;">
      <img src="./cfg_1/sample_2.png" style="width: 100%; height: auto;" alt="img3">
      <p style="margin: 4px 0 0; font-size: 14px;">图 3</p>
    </div>
    <div style="flex: 1; max-width: 120px; text-align: center;">
      <img src="./cfg_1/sample_3.png" style="width: 100%; height: auto;" alt="img4">
      <p style="margin: 4px 0 0; font-size: 14px;">图 4</p>
    </div>
  </div>
</div>

<div style="margin: 24px 0;">
  <div style="text-align: center; font-size: 16px; font-weight: bold; margin-bottom: 12px;">no_cfg: 随机生成</div>
  <div style="display: flex; justify-content: center; gap: 12px; align-items: center; margin: 16px 0;">
    <div style="flex: 1; max-width: 120px; text-align: center;">
      <img src="./no_cfg/sample_0.png" style="width: 100%; height: auto;" alt="img1">
      <p style="margin: 4px 0 0; font-size: 14px;">图 1</p>
    </div>
    <div style="flex: 1; max-width: 120px; text-align: center;">
      <img src="./no_cfg/sample_1.png" style="width: 100%; height: auto;" alt="img2">
      <p style="margin: 4px 0 0; font-size: 14px;">图 2</p>
    </div>
    <div style="flex: 1; max-width: 120px; text-align: center;">
      <img src="./no_cfg/sample_2.png" style="width: 100%; height: auto;" alt="img3">
      <p style="margin: 4px 0 0; font-size: 14px;">图 3</p>
    </div>
    <div style="flex: 1; max-width: 120px; text-align: center;">
      <img src="./no_cfg/sample_3.png" style="width: 100%; height: auto;" alt="img4">
      <p style="margin: 4px 0 0; font-size: 14px;">图 4</p>
    </div>
  </div>
</div>
